In [1]:
import yfinance as yf

In [4]:
def fetch_financial_news(query: str) -> str:
    ticker = yf.Ticker(query)
    news = ticker.news
    if not news:
        return f"No recent news found for {query}."
    
    formatted_news = []
    for article in news[:3]:
        title = article.get('title') or article.get('content', {}).get('title', 'Headline Unavailable')
        publisher = article.get('publisher') or article.get('content', {}).get('provider', {}).get('displayName', 'Unknown Publisher')
        
        formatted_news.append(f"- {title} ({publisher})")

    return f"Latest headlines for {query}:\n" + "\n".join(formatted_news)

In [5]:
print(fetch_financial_news("AAPL"))

Latest headlines for AAPL:
- Why Apple May Be the Safest AI Stock Nobody Calls an AI Stock (24/7 Wall St.)
- Apple's Foldable iPhone Is About to Face a China Problem (GuruFocus.com)
- Apple’s Sept. 9 Event Could Change Everything (24/7 Wall St.)


In [6]:
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    query: str = Field(description="The stock ticker symbol to search for (e.g., AAPL, TSLA).")

In [7]:
test_schema = SearchInput(query="NVDA")
print(test_schema.model_dump())

{'query': 'NVDA'}


In [9]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

print("Key loaded successfully" if api_key else "Key not found. Check your .env file.")

Key loaded successfully


In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool


@tool("financial_search", args_schema=SearchInput)
def financial_search_tool(query: str) -> str:
    """Fetches real-time financial headlines for a given stock ticker."""
    return fetch_financial_news(query)

llm = ChatOpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    model="gemini-3.5-flash" 
)

llm_with_tools = llm.bind_tools([financial_search_tool])

response = llm_with_tools.invoke("What is the latest news regarding Microsoft stock?")
print("Tool Calls Triggered:", response.tool_calls)

Tool Calls Triggered: [{'name': 'financial_search', 'args': {'query': 'MSFT'}, 'id': 'call_2374069', 'type': 'tool_call'}]
